# 1.1. Khám phá ID, Giao thức Atom/OAI-PMH và Lịch sử Phiên bản

## 1. Kiến trúc Discovery thích ứng chuẩn arXiv 2025–2026

1. **Truy vấn lô trực tiếp qua Atom XML (`export.arxiv.org/api/query`)**: Thay vì phụ thuộc vào các hàm tải nguồn cũ của `arxiv.py` (vốn đã trải qua thay đổi lớn ở nhánh 4.x), module `discovery.py` dùng `requests.Session` truy vấn theo lô `id_list` từ `50–100` ID/lần với nhịp nghỉ tối thiểu `≥ 3,0 giây`, kiểm tra chủ động các mục lỗi ẩn `api/errors` trong feed XML và bóc tách hậu tố phiên bản mới nhất `<id>...vN</id>` để sinh đầy đủ danh sách `v1..vN` cho từng bài báo.
2. **Tìm kiếm nhị phân biên tháng & Endpoint OAI-PMH mới `https://oaipmh.arxiv.org/oai` (cập nhật 03/2025)**: Hai phân vùng thành viên vắt qua ranh giới tháng (`2412 -> 2501` của `DAT` và `2406 -> 2407` của `TRI`). Hàm `id_range.py:find_last_id_of_month` thực hiện tìm kiếm nhị phân số thứ tự cuối tháng và gọi `discovery.py:confirm_missing_id` lên **endpoint OAI-PMH mới `https://oaipmh.arxiv.org/oai` (`metadataPrefix=arXivRaw`, thay thế `export.arxiv.org/oai2` sau đợt chuyển đổi hạ tầng 03/2025)** để phân biệt chắc chắn mã `idDoesNotExist` với lỗi mạng tạm thời, đồng thời đối soát mốc ngày nộp từng phiên bản `v1..vN` trong `version_history.py`.

## 2. Phân bố số phiên bản và giải thích toán học tỷ lệ khuyết trường

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
hist_s = pd.Series({int(k): int(v) for k, v in m1["version_histogram"].items()}).sort_index()
df_hist = pd.DataFrame({
    "Số version / bài": hist_s.index,
    "Số bài báo": hist_s.values,
})
df_hist["Tỷ lệ (%)"] = 100.0 * df_hist["Số bài báo"] / scope["papers"]
df_hist["Tích lũy (%)"] = df_hist["Tỷ lệ (%)"].cumsum()
display(style_table(
    df_hist,
    caption="Bảng 1.1. Phân bố số phiên bản (revision) trên 25.000 bài báo (nhấp tiêu đề cột để sắp xếp hoặc dùng ô lọc).",
    int_cols=["Số version / bài", "Số bài báo"],
    pct_cols=["Tỷ lệ (%)", "Tích lũy (%)"],
    bar_cols={"Số bài báo": ("var(--ah-bar-fill)", None)},
))

fig, ax1 = plt.subplots(figsize=(10.2, 4.3))
x = np.arange(len(df_hist))
bars = ax1.bar(x, df_hist["Số bài báo"], color=BOOK_PALETTE["navy"], width=0.66, label="Số bài báo")
ax1.set_xticks(x); ax1.set_xticklabels(df_hist["Số version / bài"].astype(str), rotation=0)
ax1.set_xlabel("Số phiên bản trên một bài báo (vN)"); ax1.set_ylabel("Số bài báo")
ax1.set_ylim(0, df_hist["Số bài báo"].max() * 1.18); ax1.grid(axis="y")
for idx_b, (b, cnt, pct) in enumerate(zip(bars, df_hist["Số bài báo"], df_hist["Tỷ lệ (%)"])):
    if idx_b < 5:
        ax1.text(b.get_x() + b.get_width()/2, cnt + 220, f"{fmt_int(cnt)}\n({fmt_pct(pct, 1)})",
                 ha="center", va="bottom", fontsize=7.8, fontweight="bold")

ax2 = ax1.twinx()
ax2.spines["top"].set_visible(False)
ax2.plot(x, df_hist["Tích lũy (%)"], color=BOOK_PALETTE["amber"], marker="o", lw=2.0, markersize=4.5, label="CDF tích lũy (%)")
ax2.set_ylabel("Tỷ lệ tích lũy (%)", color=BOOK_PALETTE["amber"]); ax2.set_ylim(50, 103)

# Khung phóng đại log-scale cho đuôi dài v >= 5..17
ins = ax1.inset_axes([0.46, 0.36, 0.46, 0.46])
tail = df_hist[df_hist["Số version / bài"] >= 5]
tx = np.arange(len(tail))
ins.bar(tx, tail["Số bài báo"].to_numpy(dtype=float), color=BOOK_PALETTE["teal"], width=0.62)
ins.set_xticks(tx); ins.set_xticklabels([str(int(v)) for v in tail["Số version / bài"]])
ins.set_yscale("log"); ins.set_title("Phóng đại đuôi dài v = 5..17 (thang log)", fontsize=8.3)
ins.tick_params(labelsize=7.6)
for i_t, v_t in enumerate(tail["Số bài báo"]):
    ins.text(i_t, float(v_t) * 1.18, str(int(v_t)), ha="center", fontsize=7.2, fontweight="bold")

ax1.set_title("Hình 1.1. Phân bố Pareto số phiên bản/bài báo và đường tích lũy CDF (trung bình 1,66 phiên bản/bài)")
plt.tight_layout(); plt.show()

In [ ]:
miss_m1 = pd.Series(m1["metadata_missing_fraction"])
df_meta = pd.DataFrame({
    "Trường metadata": miss_m1.index,
    "Tỷ lệ thu thập đầy đủ (%)": (1.0 - miss_m1.values) * 100.0,
    "Tỷ lệ khuyết / null (%)": miss_m1.values * 100.0,
})
display(style_table(
    df_meta,
    caption="Bảng 1.2. Độ đầy đủ các trường metadata cấp bài báo trên 25.000 bài.",
    pct_cols=["Tỷ lệ thu thập đầy đủ (%)", "Tỷ lệ khuyết / null (%)"],
    bar_cols={"Tỷ lệ thu thập đầy đủ (%)": ("rgba(5, 150, 105, 0.22)", 100.0)},
))

fig, ax = plt.subplots(figsize=(9.4, 2.9))
y = np.arange(len(df_meta))
ax.barh(y, df_meta["Tỷ lệ thu thập đầy đủ (%)"], color=BOOK_PALETTE["emerald"], height=0.52, label="Có giá trị (%)")
ax.barh(y, df_meta["Tỷ lệ khuyết / null (%)"], left=df_meta["Tỷ lệ thu thập đầy đủ (%)"],
        color="#E2E8F0", edgecolor="#94A3B8", height=0.52, label="Null trên Atom feed (%)")
ax.set_yticks(y); ax.set_yticklabels(df_meta["Trường metadata"])
ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel("Tỷ lệ (%)")
ax.set_title("Hình 1.2. Độ đầy đủ metadata cấp bài báo (preprint 06/2024 – 02/2025)")
ax.legend(loc="lower right", frameon=True, fontsize=8.3)
for idx_m, row_m in enumerate(df_meta.to_dict("records")):
    full_v = row_m["Tỷ lệ thu thập đầy đủ (%)"]
    if full_v > 50:
        ax.text(full_v / 2, idx_m, fmt_pct(full_v), ha="center", va="center", color="white", fontweight="bold", fontsize=8.4)
    else:
        ax.text(full_v + 2.0, idx_m, f"Đủ {fmt_pct(full_v)} | Null {fmt_pct(row_m['Tỷ lệ khuyết / null (%)'])}",
                va="center", color="#1E293B", fontweight="bold", fontsize=8.3)
plt.tight_layout(); plt.show()

```{tip}
**Chứng minh đại số cho `5.565` mốc ngày phiên bản trung gian mang giá trị `null`:**
Trên `25.000` bài báo có đúng `14.067` bài đơn phiên bản (`v1` nhận `atom:published`) và `10.933` bài đa phiên bản (`v1` nhận `published`, `vN` nhận `updated`). Tổng số mốc thời gian Atom feed cung cấp trực tiếp là $14.067 \times 1 + 10.933 \times 2 = 35.933$ phiên bản $\Rightarrow$ phần chênh $41.498 - 35.933 = \mathbf{5.565}$ khớp chính xác 100% với các phiên bản trung gian $v_2 \dots v_{N-1}$ của `3.667` bài có từ `3` revision trở lên (pipeline giữ `null` trung thực, tuyệt đối không nội suy giả). Tương tự, `doi` khuyết `77,70%` và `venue` khuyết `82,54%` phản ánh đặc trưng tự nhiên của các bản thảo AI/CS mới nộp giai đoạn `2024–2025`.
```

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/id_range.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/id_range.py).
- Mã nguồn triển khai: [`src/crawler/discovery.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/discovery.py).
- Mã nguồn triển khai: [`src/crawler/version_history.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/version_history.py).
- Mã nguồn triển khai: [`src/crawler/request_policy.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/request_policy.py).
- Mã nguồn triển khai: [`src/crawler/rate_limiter.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/rate_limiter.py).